# NEOFC - Run MapConn analysis to generate NEOFC outcomes

In [ ]:
from pathlib import Path 
import pandas as pd 
import numpy as np 
import xarray as xr 
import pickle
import gzip 

from tqdm.auto import tqdm

from utils import get_ref_data, get_dist_mat, get_stats
from mapconn import MapConn, MapConnNull, MapConnInv

from nispace.stats.misc import residuals_nan

# working directory
wd = Path.cwd() 
print(wd)

# general vars
from utils import (PARCS_ALL, PARCS_CX, PARC_DEFAULT,
                   MEG_MEASURES_ALL, MEG_FQBANDS)


/Users/llotter/projects/mapfc


##  Data

### Reference data

In [2]:
data_ref_pet = get_ref_data(ref="pet", parcs=PARCS_ALL)
data_ref_rsn = get_ref_data(ref="rsn", parcs=PARCS_CX)
data_ref_cov = get_ref_data(ref="cov", parcs=PARCS_CX)

data_ref_pet[PARC_DEFAULT].head()

Loading parcellated pet data, standardized=True, null=False
Loading parcellated rsn data, standardized=True, null=False
Loading parcellated cov data, standardized=True, null=False


,hemi-L_div-Vis_lab-1,hemi-L_div-Vis_lab-2,hemi-L_div-Vis_lab-3,hemi-L_div-Vis_lab-4,hemi-L_div-Vis_lab-5,hemi-L_div-Vis_lab-6,hemi-L_div-Vis_lab-7,hemi-L_div-Vis_lab-8,hemi-L_div-Vis_lab-9,hemi-L_div-Vis_lab-10,...,hemi-R_div-Default_lab-PFCm+1,hemi-R_div-Default_lab-PFCm+2,hemi-R_div-Default_lab-PFCm+3,hemi-R_div-Default_lab-PFCm+4,hemi-R_div-Default_lab-PFCm+5,hemi-R_div-Default_lab-PFCm+6,hemi-R_div-Default_lab-PFCm+7,hemi-R_div-Default_lab-PCC+1,hemi-R_div-Default_lab-PCC+2,hemi-R_div-Default_lab-PCC+3
5HT1a,0.242521,-1.012039,0.103727,-1.298651,-1.664068,-0.337140,-2.487159,0.259029,-1.484782,-1.746785,...,0.816689,0.829252,-0.365888,0.245154,-0.079351,-0.149584,-0.495787,-0.831653,-0.278468,-0.442881
5HT1b,-1.032709,-0.281828,-0.142056,0.009959,0.558745,-2.702971,3.068384,0.249281,0.431451,2.601738,...,-0.701319,-0.064954,-1.336322,0.263009,0.592660,0.781236,0.734859,-0.837514,-0.154833,-0.098862
5HT2a,-0.046030,-0.334491,0.437479,-0.454518,-0.104702,-2.292043,0.141766,0.735451,-0.037308,0.584442,...,-0.180471,-0.357249,-2.654473,0.467877,0.663986,0.560142,0.404215,-0.752663,1.199359,0.435919
5HT4,-0.031954,-0.744829,0.573733,-1.319910,-1.706253,-1.136569,-2.265509,0.746292,-1.083537,-1.376613,...,0.384866,0.120564,-1.304104,0.096612,-0.027086,-0.252184,-0.504667,-0.281662,0.843254,0.506718
5HT6,-0.375488,-0.142426,0.595812,-1.227212,-0.058230,-3.703124,0.312230,0.956246,0.184458,0.804090,...,-0.329729,0.089328,-1.763844,0.612178,0.108492,0.909910,0.582685,-0.601393,0.601853,1.036586


### Null reference data

In [3]:
data_ref_pet_null = get_ref_data(ref="pet", parcs=PARCS_ALL, null=True)
data_ref_rsn_null = get_ref_data(ref="rsn", parcs=PARCS_CX, null=True)
data_ref_cov_null = get_ref_data(ref="cov", parcs=PARCS_CX, null=True)

Loading parcellated pet data, standardized=True, null=True
Loading parcellated rsn data, standardized=True, null=True
Loading parcellated cov data, standardized=True, null=True


## General functions

In [4]:
# number of permutations
n_perm = 1000

# number of jobs
n_jobs = -1

# overwrite
overwrite = "stats_only"

In [5]:
def mapconn_from_connectivity(
    connectivity_xarray, parc_name, 
    ref_name="pet", 
    ref_maps=None, 
    n_perm=n_perm, 
    n_jobs=n_jobs, 
    drop_nulls=False, 
    drop_nulls_keep_stats=True,
    run_nointerhemispheric=True, 
    run_null=True, 
    run_delta_null=False,
    measures=["pearson", "aec"],
    save_path=None, 
    save_path_stats=None,
    stats_to_save=["auc"],
    save_delta=False,
    index_levels=None,
    overwrite=True
):
    """Function to compute MapConn from connectivity xarray"""
    
    # check stats settings
    if not isinstance(stats_to_save, list):
        stats_to_save = [stats_to_save]
    if save_path_stats is not None and index_levels is None:
        raise ValueError("index_levels must be provided if save_path_stats is provided")
    
    # save paths
    if save_path is not None:
        save_path = Path(save_path)
    if save_path_stats is not None:
        save_path_stats = Path(save_path_stats)
    
    # stats save function
    def save_stats(results, stat_to_save, index_levels, fp):
        print(f"Saving stats to: {fp} (stat: {stat_to_save}, save_delta: {save_delta})")
        fp = Path(fp)
        exts = [".csv.gz", ".csv", ".tsv.gz", ".tsv"]
        name = None
        for ext in exts:
            if fp.name.endswith(ext):
                name = fp.name.replace(ext, "")
                break
        if name is None:
            raise ValueError(f"Could not identify extension of file path: {fp}")
        sep = "\t" if "tsv" in ext else ","
        
        out = get_stats(results, stat=stat_to_save, levels=["measure", "connections"] + index_levels,
                        recalculate_dist_stats=False, get_nulls_stats=run_null, 
                        drop_delta=not save_delta)
        if len(out) == 2:
            df_group, df_individual = out
        elif len(out) == 4:
            df_group, df_individual, df_indiv_nulls, df_group_nulls = out
        else:
            raise ValueError(f"Unexpected number of outputs: {len(out)}")
        df_group.to_csv(fp.parent / f"{name}_stat-{stat_to_save}_group{ext}", sep=sep)
        df_individual.to_csv(fp.parent / f"{name}_stat-{stat_to_save}_individual{ext}", sep=sep)
        if run_null:
            df_indiv_nulls.to_csv(fp.parent / f"{name}_stat-{stat_to_save}_individual_nulls{ext}", sep=sep)
            df_group_nulls.to_csv(fp.parent / f"{name}_stat-{stat_to_save}_group_nulls{ext}", sep=sep)
    
    # check overwrite
    if save_path is not None:
        save_path = Path(save_path)
        if save_path.exists():
            print(f"File {save_path} exists.")
            if not overwrite:
                print("Overwrite is False. Returning None")
                return None
            elif overwrite == True:
                print("Overwrite is True. Starting analysis")
            elif overwrite == "stats_only":
                print("Overwrite is stats_only. Loading mapconn data and overwriting stats.")
                
                # load mapconn data
                results = pickle.load(gzip.open(save_path, "rb"))
                
                # save stats
                if save_path_stats is not None:
                    for stat in stats_to_save:
                        save_stats(results, stat, index_levels, save_path_stats)
                return results
            
            else:
                raise ValueError(f"Overwrite must be True, False, or stats_only. Got {overwrite}")
                
    else:
        print("No save path provided. Sure this is intentional?")
    
    # check arr
    if not isinstance(connectivity_xarray, xr.DataArray):
        raise ValueError("connectivity_xarray must be an xarray DataArray")
    print("connectivity_xarray.shape:", connectivity_xarray.shape)
    xarray_dims = connectivity_xarray.dims
    xarray_coords = list(connectivity_xarray.coords)
    print("xarray_dims:", xarray_dims)
    print("xarray_coords:", xarray_coords)  
    # we expect min coords: measure, sub, parcelx, parcely
    if not all(k in xarray_coords for k in ["measure", "sub", "parcelx", "parcely"]):
        raise ValueError("connectivity_xarray must have 'measure', 'sub', 'parcelx' and 'parcely' coordinates")
    
    # get measures
    conn_measures = connectivity_xarray.coords["measure"].values.tolist()
    print("conn_measures:", conn_measures)
    conn_measures = [m for m in conn_measures if m in measures]
    print("retaining conn_measures:", conn_measures)
    
    # index to iterate over
    try:
        index_to_iterate = connectivity_xarray.coords[xarray_dims[1]].to_pandas().index.droplevel("sub").unique()
        print("index_to_iterate:", index_to_iterate[:5])
    except ValueError:
        index_to_iterate = None
        print("No index to iterate")
    
    # check parc
    if parc_name not in PARCS_ALL:
        raise ValueError(f"Parc {parc_name} not supported")
    
    # get ref data
    if isinstance(ref_name, pd.DataFrame):
        data_ref = ref_name
        data_ref_null = None
    elif isinstance(ref_name, str):
        if ref_name not in ["pet", "rsn", "cov"]:
            raise ValueError(f"Ref {ref_name} not supported")
        data_ref = { "pet": data_ref_pet, "rsn": data_ref_rsn, "cov": data_ref_cov }[ref_name][parc_name]
        data_ref_null = { "pet": data_ref_pet_null, "rsn": data_ref_rsn_null, "cov": data_ref_cov_null }[ref_name][parc_name]
        
    # ref maps
    if ref_maps is not None:
        if not all([m in data_ref.index for m in ref_maps]):
            raise ValueError("Not all ref_maps fround in ref data!")
        data_ref = data_ref.loc[ref_maps]
        
    # dict to store mapconn results
    results = {}
    
    # calculate mapconn independently for measures, hemispheres, and all idc_names_to_iterate
    for measure in conn_measures:
        results[measure] = {}
        
        # iterate connections settings
        for connections in ["all", "nointer"]:
            if connections == "nointer" and not run_nointerhemispheric:
                continue
            results[measure][connections] = {}
                
            def proc_fun(conn):
                
                # get subjects
                subs = conn.coords["sub"].values.tolist()
                
                # set corss-hemisphere blocks to non for nointer
                # NOTE: only works with symmetric length distance matrices
                if connections == "nointer":
                    n_parcel = conn.shape[-1]
                    # lower left block to nan
                    conn[:, n_parcel//2:, :n_parcel//2] = np.nan
                    # upper right block to nan
                    conn[:, :n_parcel//2, n_parcel//2:] = np.nan
                
                # calculate mapconn
                mapconn = MapConn.from_matrix(
                    connectivity_matrices=conn.values,
                    matrix_ids=subs,
                    map_data=data_ref,
                    r_to_z=False,
                    n_jobs=n_jobs,
                    dtype=np.float32,
                    verbose=True
                )
                
                # calculate inverse mapconn
                mapconn_inv = MapConnInv.from_mapconn(mapconn_instance=mapconn)
                
                if run_null:
                    # calculate null mapconn
                    mapconn_null = MapConnNull.from_mapconn(
                        mapconn_instance=mapconn_inv,
                        map_data_null=data_ref_null,
                        seed=42,
                        n_nulls=n_perm,
                        n_jobs=-1,
                        verbose=True,
                        null_verbose=False
                    )
                    
                    # calculate results
                    mapconn_null._ensure_results(include_delta=run_delta_null, stats=stats_to_save)
                    
                    if drop_nulls:
                        print(f"Dropping nulls. Keeping null stats: {drop_nulls_keep_stats}")
                        mapconn_null.drop_nulls(keep_null_stats=drop_nulls_keep_stats, stats=stats_to_save)
                        
                return mapconn_null if run_null else mapconn_inv
            
            if index_to_iterate is not None:
            
                # iterate indices
                for idc in tqdm(index_to_iterate, desc=f"Processing {measure}"):
                    if not isinstance(idc, tuple):
                        idc = (idc,)
                    
                    # get connectivity matrices
                    conn = connectivity_xarray.loc[measure, :, :, :].loc[(slice(None), *idc), :, :]
                    print(f"{measure} | {connections} | idc = {idc}:", conn.shape)
                    mapconn = proc_fun(conn)
                        
                    # store results
                    print("Storing results")
                    if len(idc) == 1:
                        idc = idc[0]
                    results[measure][connections][idc] = mapconn
            
            else:
                
                # get connectivity
                print("Processing measure")
                conn = connectivity_xarray.loc[measure, :, :, :]
                print(f"{measure} | {connections}:", conn.shape)
                mapconn = proc_fun(conn)
                
                # store results
                print("Storing results")
                results[measure][connections] = mapconn
                
    # save mapconn
    if save_path is not None:
        print(f"Saving results to: {save_path}")
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with gzip.open(save_path, "wb", compresslevel=9) as f:
            pickle.dump(results, f)
            
    # save stats
    if save_path_stats is not None:
        save_path_stats.parent.mkdir(parents=True, exist_ok=True)
        for stat in stats_to_save:
            save_stats(results, stat, index_levels, save_path_stats)
    
    # return results
    return results


## HCP-YA: MRI

### RSN

In [6]:
# Resting-State Networks
for parc in PARCS_CX:
    print("RSN:", parc)
    
    # load connectivity
    conn_ya_mri = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ya_mri" / f"parc-{parc}_connectivity.pkl.gz", "rb"))
    
    # run
    mapconn_ya_mri_rsn = mapconn_from_connectivity(  
        conn_ya_mri, 
        parc, 
        ref_name="rsn",
        n_jobs=n_jobs,
        run_null=True,
        run_delta_null=False,
        n_perm=n_perm,
        run_nointerhemispheric=False,
        drop_nulls=True,
        save_path=wd / "data_deriv" / "mapconn_pickled" / "hcp_ya_mri" / f"parc-{parc}_dset-rsn_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "hcp_ya_mri" / f"parc-{parc}_dset-rsn.csv.gz",
        index_levels=["run"],
        stats_to_save=["auc", "poly2"],
        overwrite=overwrite
    )
    
del conn_ya_mri, mapconn_ya_mri_rsn

RSN: Schaefer100
File /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/hcp_ya_mri/parc-Schaefer100_dset-rsn_mapconn.pkl.gz exists.
Overwrite is stats_only. Loading mapconn data and overwriting stats.
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer100_dset-rsn.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer100_dset-rsn.csv.gz (stat: poly2, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)
RSN: Schaefer200
File /Users/llotter/p

### PET

In [7]:
# PET
for parc in PARCS_ALL:
    print(f"PET: {parc}")
        
    # load connectivity
    conn_ya_mri = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ya_mri" / f"parc-{parc}_connectivity.pkl.gz", "rb"))
    
    # run
    mapconn_ya_mri_pet = mapconn_from_connectivity(  
        conn_ya_mri, 
        parc, 
        n_jobs=n_jobs,
        run_null=True,
        run_delta_null=True,
        n_perm=n_perm,
        run_nointerhemispheric=True if PARC_DEFAULT in parc else False,
        measures=["pearson"],        
        drop_nulls=True,#False if parc == PARC_DEFAULT else True,      
        save_path=wd / "data_deriv" / "mapconn_pickled" / "hcp_ya_mri" / f"parc-{parc}_dset-pet_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "hcp_ya_mri" / f"parc-{parc}_dset-pet.csv.gz",
        save_delta=True,
        index_levels=["run"],
        stats_to_save=["auc", "poly2"],
        overwrite=overwrite
    )
    
del conn_ya_mri, mapconn_ya_mri_pet

PET: Schaefer100
File /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/hcp_ya_mri/parc-Schaefer100_dset-pet_mapconn.pkl.gz exists.
Overwrite is stats_only. Loading mapconn data and overwriting stats.
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer100_dset-pet.csv.gz (stat: auc, save_delta: True)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer100_dset-pet.csv.gz (stat: poly2, save_delta: True)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)
PET: Schaefer100Subcortical
File /Users/

Processing pearson:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)


pearson | all | idc = ('1',): (112, 200, 200)


INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:08<00:00, 59.35it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 1559.85it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<

...

pearson | all | idc = ('1',): (112, 416, 416)


Calculating mapConn curves: 100%|██████████| 500/500 [00:02<00:00, 189.73it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 416 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:02<00:00, 209.98it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 1000/

Dropping nulls. Keeping null stats: True
Storing results
pearson | all | idc = ('2',): (112, 416, 416)


INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 416 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:02<00:00, 180.92it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 416 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:02<00:00, 175.84it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<

Dropping nulls. Keeping null stats: True
Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/hcp_ya_mri/parc-Schaefer400Subcortical_dset-pet_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer400Subcortical_dset-pet.csv.gz (stat: auc, save_delta: True)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer400Subcortical_dset-pet.csv.gz (stat: poly2, save_delta: True)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)


### PET: significance in lower percentiles

In [8]:
# iterate parcs
for parc in ["Schaefer200Subcortical", "Schaefer200"]:
    print(parc)
        
    # load connectivity
    conn_ya_mri = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ya_mri" / f"parc-{parc}_connectivity.pkl.gz", "rb"))

    # run
    mapconn_ya_mri_aucthresh = mapconn_from_connectivity(  
        conn_ya_mri, 
        parc, 
        n_jobs=n_jobs,
        run_null=True,
        run_delta_null=False,
        n_perm=n_perm,
        run_nointerhemispheric=False,
        measures=["pearson"],        
        drop_nulls=True,
        save_path=wd / "data_deriv" / "mapconn_pickled" / "hcp_ya_mri" / f"parc-{parc}_dset-petaucthresh_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "hcp_ya_mri" / f"parc-{parc}_dset-petaucthresh.csv.gz",
        index_levels=["run"],
        overwrite=overwrite,
        stats_to_save=[f"auc<={i}" for i in range(5, 100, 5)]
    )

    del conn_ya_mri, mapconn_ya_mri_aucthresh

Schaefer200Subcortical
File /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/hcp_ya_mri/parc-Schaefer200Subcortical_dset-petaucthresh_mapconn.pkl.gz exists.
Overwrite is stats_only. Loading mapconn data and overwriting stats.
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer200Subcortical_dset-petaucthresh.csv.gz (stat: auc<=5, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer200Subcortical_dset-petaucthresh.csv.gz (stat: auc<=10, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting stati

### NET: Covariates

In [9]:
# Resting-State Networks
parc = "Schaefer200"

# load connectivity
conn_ya_mri = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ya_mri" / f"parc-{parc}_connectivity.pkl.gz", "rb"))

# run
mapconn_ya_mri_cov = mapconn_from_connectivity(  
    conn_ya_mri, 
    parc, 
    ref_name="cov",
    n_jobs=n_jobs,
    run_null=True,
    run_delta_null=False,
    n_perm=n_perm,
    run_nointerhemispheric=False,
    measures=["pearson"],  
    drop_nulls=True,
    save_path=wd / "data_deriv" / "mapconn_pickled" / "hcp_ya_mri" / f"parc-{parc}_dset-cov_mapconn.pkl.gz",
    save_path_stats=wd / "results" / "neofc" / "hcp_ya_mri" / f"parc-{parc}_dset-cov.csv.gz",
    index_levels=["run"],
    overwrite=overwrite
)

del conn_ya_mri, mapconn_ya_mri_cov

File /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/hcp_ya_mri/parc-Schaefer200_dset-cov_mapconn.pkl.gz exists.
Overwrite is stats_only. Loading mapconn data and overwriting stats.
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_mri/parc-Schaefer200_dset-cov.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)


## HCP-YA: MEG

In [ ]:
for parc in PARCS_CX:
    print(parc)
    
    # load connectivity
    conn_ya_meg = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ya_meg" / f"parc-{parc}_connectivity.pkl.gz", "rb"))
    if parc == "Schaefer200":
        conn_ya_meg = conn_ya_meg.loc[["aec", "aecorth"], :, :, :]
    # measures
    if parc == "Schaefer100":
        measures = ["aec"]
    elif parc == "Schaefer200":
        measures = ["aec", "aecorth"]
    elif parc == "Schaefer400":
        measures = ["aec"]
    
    # calculate mapconn
    mapconn_ya_meg = mapconn_from_connectivity(
        conn_ya_meg, parc, n_perm=n_perm, n_jobs=n_jobs, 
        measures=measures,
        run_nointerhemispheric=False,
        run_delta_null=False, 
        drop_nulls=True,
        save_path=wd / "data_deriv" / "mapconn_pickled" / "hcp_ya_meg" / f"parc-{parc}_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "hcp_ya_meg" / f"parc-{parc}.csv.gz",
        index_levels=["fqband"],
        overwrite=overwrite
    )
    
del conn_ya_meg, mapconn_ya_meg

Schaefer100
connectivity_xarray.shape: (2, 198, 100, 100)
xarray_dims: ('measure', 'sub-fqband', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-fqband', 'sub', 'fqband', 'parcelx', 'parcely']
conn_measures: ['aec', 'aecorth']
retaining conn_measures: ['aec']
index_to_iterate: Index(['delta', 'theta', 'alpha', 'beta', 'lgamma'], dtype='object', name='fqband')


Processing aec:   0%|          | 0/6 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 100 values
INFO:mapconn.percentiles:63: Calculating map percentiles


aec | all | idc = ('delta',): (33, 100, 100)


Calculating mapConn curves: 100%|██████████| 500/500 [00:06<00:00, 72.71it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 100 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 5430.48it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 1000/

...

Dropping nulls. Keeping null stats: True
Storing results
aec | all | idc = ('lgamma',): (33, 400, 400)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 1067.67it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 400 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 939.49it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 1000

Dropping nulls. Keeping null stats: True
Storing results
aec | all | idc = ('hgamma',): (33, 400, 400)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 913.49it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 400 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 1045.22it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 1000

Dropping nulls. Keeping null stats: True
Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/hcp_ya_meg/parc-Schaefer400_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_meg/parc-Schaefer400.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level fqband (should be run, ses, or treat?)
Getting statistics for: ('aec', 'all', 'delta')
Getting statistics for: ('aec', 'all', 'theta')
Getting statistics for: ('aec', 'all', 'alpha')
Getting statistics for: ('aec', 'all', 'beta')
Getting statistics for: ('aec', 'all', 'lgamma')
Getting statistics for: ('aec', 'all', 'hgamma')
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ya_meg/parc-Schaefer400.csv.gz (stat: poly2, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level fqband (should b

## DRUG: mph

In [11]:
parc = "Schaefer200"
print(parc)

# load connectivity
conn_mph = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "drug_mph" / f"parc-{parc}_connectivity.pkl.gz", "rb"))

# calculate mapconn
mapconn_mph = mapconn_from_connectivity(
    conn_mph, 
    parc, 
    n_jobs=n_jobs, 
    run_null=True,
    run_delta_null=False,
    n_perm=n_perm,
    run_nointerhemispheric=False,
    drop_nulls=True,
    measures=["pearson"],
    save_path=wd / "data_deriv" / "mapconn_pickled" / "drug_mph" / f"parc-{parc}_mapconn.pkl.gz",
    save_path_stats=wd / "results" / "neofc" / "drug_mph" / f"parc-{parc}.csv.gz",
    index_levels=["treat"],
    overwrite=True
)

del conn_mph, mapconn_mph

Schaefer200
connectivity_xarray.shape: (1, 60, 200, 200)
xarray_dims: ('measure', 'sub-treat', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-treat', 'sub', 'treat', 'parcelx', 'parcely']
conn_measures: ['pearson']
retaining conn_measures: ['pearson']
index_to_iterate: Index(['placebo', 'mph'], dtype='object', name='treat')


Processing pearson:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles


pearson | all | idc = ('placebo',): (30, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 2970.67it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3105.76it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
pearson | all | idc = ('mph',): (30, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3329.17it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3163.90it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/drug_mph/parc-Schaefer200_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/drug_mph/parc-Schaefer200.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level treat (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 'placebo')
Getting statistics for: ('pearson', 'all', 'mph')


## DRUG: risp

In [12]:
# load connectivity for Schaefer200Subcortical
conn_risp = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "drug_risp" / f"parc-Schaefer200Subcortical_connectivity.pkl.gz", "rb"))

# iterate parcs
for parc in ["Schaefer200Subcortical", "Schaefer200"]:
    print(parc)
    
    # if parc is Schaefer200, we drop subcortical parcels
    if parc == "Schaefer200":
        conn_risp = conn_risp[:, :, :200, :200]

    # calculate mapconn
    mapconn_risp = mapconn_from_connectivity(
        conn_risp, 
        parc, 
        n_jobs=n_jobs, 
        run_null=True,
        run_delta_null=False,
        n_perm=n_perm,
        run_nointerhemispheric=False,
        measures=["pearson"],
        save_path=wd / "data_deriv" / "mapconn_pickled" / "drug_risp" / f"parc-{parc}_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "drug_risp" / f"parc-{parc}.csv.gz",
        index_levels=["treat"],
        drop_nulls=True,
        overwrite=overwrite
    )

del conn_risp, mapconn_risp

Schaefer200Subcortical
connectivity_xarray.shape: (1, 53, 216, 216)
xarray_dims: ('measure', 'sub-treat', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-treat', 'sub', 'treat', 'parcelx', 'parcely']
conn_measures: ['pearson']
retaining conn_measures: ['pearson']
index_to_iterate: Index(['placebo', 'risp_low', 'risp_high'], dtype='object', name='treat')


Processing pearson:   0%|          | 0/3 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles


pearson | all | idc = ('placebo',): (17, 216, 216)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 4120.29it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3584.10it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

...

Dropping nulls. Keeping null stats: True
Storing results
pearson | all | idc = ('risp_low',): (19, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 4581.01it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 5329.80it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
pearson | all | idc = ('risp_high',): (17, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 4140.75it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 5400.80it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/drug_risp/parc-Schaefer200_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/drug_risp/parc-Schaefer200.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level treat (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 'placebo')
Getting statistics for: ('pearson', 'all', 'risp_low')
Getting statistics for: ('pearson', 'all', 'risp_high')


## DRUG: ket

### Drug vs Placebo

In [13]:
# load connectivity for Schaefer200Subcortical
conn_ket = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "drug_ket" / f"parc-Schaefer200Subcortical_connectivity.pkl.gz", "rb"))

# iterate parcs
for parc in ["Schaefer200Subcortical", "Schaefer200"]:
    print(parc)
    
    # if parc is Schaefer200, we drop subcortical parcels
    if parc == "Schaefer200":
        conn_ket = conn_ket[:, :, :200, :200]

    # calculate mapconn
    mapconn_ket = mapconn_from_connectivity(
        conn_ket, 
        parc, 
        n_jobs=n_jobs, 
        run_null=True,
        run_delta_null=False,
        n_perm=n_perm,
        run_nointerhemispheric=False,
        measures=["pearson"],
        save_path=wd / "data_deriv" / "mapconn_pickled" / "drug_ket" / f"parc-{parc}_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "drug_ket" / f"parc-{parc}.csv.gz",
        index_levels=["treat"],
        drop_nulls=True,
        overwrite=overwrite
    )

del conn_ket, mapconn_ket

Schaefer200Subcortical
connectivity_xarray.shape: (1, 112, 216, 216)
xarray_dims: ('measure', 'sub-treat-ses', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-treat-ses', 'sub', 'treat', 'ses', 'parcelx', 'parcely']
conn_measures: ['pearson']
retaining conn_measures: ['pearson']
index_to_iterate: MultiIndex([( 'placebo',  'pre'),
            ('ketamine',  'pre'),
            ( 'placebo', 'post'),
            ('ketamine', 'post')],
           names=['treat', 'ses'])


Processing pearson:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles


pearson | all | idc = ('placebo', 'pre'): (28, 216, 216)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3186.36it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3556.56it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

...

Dropping nulls. Keeping null stats: True
Storing results
pearson | all | idc = ('placebo', 'post'): (28, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3821.31it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 4008.46it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
pearson | all | idc = ('ketamine', 'post'): (28, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3503.31it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3580.63it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/drug_ket/parc-Schaefer200_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/drug_ket/parc-Schaefer200.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level treat (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', ('placebo', 'pre'))
Getting statistics for: ('pearson', 'all', ('ketamine', 'pre'))
Getting statistics for: ('pearson', 'all', ('placebo', 'post'))
Getting statistics for: ('pearson', 'all', ('ketamine', 'post'))


### Windowed

In [14]:
# load connectivity for Schaefer200Subcortical
conn_ket = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "drug_ket" / f"parc-Schaefer200Subcortical_windowconnectivity.pkl.gz", "rb"))

for parc in ["Schaefer200Subcortical", "Schaefer200"]:
    print(parc)
    
    # if parc is Schaefer200, we drop subcortical parcels
    if parc == "Schaefer200":
        conn_ket = conn_ket[:, :, :200, :200]

    # calculate mapconn
    mapconn_ket = mapconn_from_connectivity(
        conn_ket, 
        parc, 
        n_jobs=n_jobs, 
        run_null=False, 
        run_nointerhemispheric=False,
        measures=["pearson"],
        save_path=wd / "data_deriv" / "mapconn_pickled" / "drug_ket" / f"parc-{parc}_window_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "drug_ket" / f"parc-{parc}_window.csv.gz",
        index_levels=["treat"],
        overwrite=overwrite
    )

del conn_ket, mapconn_ket

Schaefer200Subcortical
connectivity_xarray.shape: (1, 4592, 216, 216)
xarray_dims: ('measure', 'sub-treat', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-treat', 'sub', 'treat', 'parcelx', 'parcely']
conn_measures: ['pearson']
retaining conn_measures: ['pearson']
index_to_iterate: Index(['placebo', 'ketamine'], dtype='object', name='treat')


Processing pearson:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)


pearson | all | idc = ('placebo',): (2296, 216, 216)


INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles
/Applications/miniforge3/envs/mapfc/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Calculating mapConn curves: 100%|██████████| 500/500 [00:20<00:00, 23.92it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentil

...

pearson | all | idc = ('placebo',): (2296, 200, 200)


INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:13<00:00, 37.95it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:14<00:00, 35.60it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<cl

Storing results
pearson | all | idc = ('ketamine',): (2296, 200, 200)


INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:14<00:00, 33.85it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:15<00:00, 31.92it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<cl

Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/drug_ket/parc-Schaefer200_window_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/drug_ket/parc-Schaefer200_window.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level treat (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 'placebo')
Getting statistics for: ('pearson', 'all', 'ketamine')


## DRUG: mdz

### Drug vs Placebo

In [15]:
# load connectivity
conn_mdz = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "drug_mdz" / f"parc-Schaefer200Subcortical_connectivity.pkl.gz", "rb"))

# iterate parcs
for parc in ["Schaefer200Subcortical", "Schaefer200"]:
    print(parc)
    
    # if parc is Schaefer200, we drop subcortical parcels
    if parc == "Schaefer200":
        conn_mdz = conn_mdz[:, :, :200, :200]
        
    # calculate mapconn
    mapconn_mdz = mapconn_from_connectivity(
        conn_mdz, 
        parc, 
        n_jobs=n_jobs, 
        run_null=False, # not needed; we only need p values for the placebo group (incl. in KET)
        run_nointerhemispheric=False,
        run_delta_null=False,
        measures=["pearson"],
        save_path=wd / "data_deriv" / "mapconn_pickled" / "drug_mdz" / f"parc-{parc}_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "drug_mdz" / f"parc-{parc}.csv.gz",
        index_levels=["treat"],
        drop_nulls=True,
        overwrite=overwrite
    )

del conn_mdz, mapconn_mdz

Schaefer200Subcortical
connectivity_xarray.shape: (1, 104, 216, 216)
xarray_dims: ('measure', 'sub-treat-ses', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-treat-ses', 'sub', 'treat', 'ses', 'parcelx', 'parcely']
conn_measures: ['pearson']
retaining conn_measures: ['pearson']
index_to_iterate: MultiIndex([(  'placebo',  'pre'),
            ('midazolam',  'pre'),
            (  'placebo', 'post'),
            ('midazolam', 'post')],
           names=['treat', 'ses'])


Processing pearson:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles


pearson | all | idc = ('placebo', 'pre'): (28, 216, 216)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3375.25it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3176.48it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class

...

Storing results
pearson | all | idc = ('placebo', 'post'): (28, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3849.36it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3830.95it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class

Storing results
pearson | all | idc = ('midazolam', 'post'): (24, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3938.06it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3927.91it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)


Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/drug_mdz/parc-Schaefer200_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/drug_mdz/parc-Schaefer200.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level treat (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', ('placebo', 'pre'))
Getting statistics for: ('pearson', 'all', ('midazolam', 'pre'))
Getting statistics for: ('pearson', 'all', ('placebo', 'post'))
Getting statistics for: ('pearson', 'all', ('midazolam', 'post'))


### Windowed

In [16]:
# load connectivity for Schaefer200Subcortical
conn_mdz = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "drug_mdz" / f"parc-Schaefer200Subcortical_windowconnectivity.pkl.gz", "rb"))

# iterate parcs
for parc in ["Schaefer200Subcortical", "Schaefer200"]:
    print(parc)
    
    # if parc is Schaefer200, we drop subcortical parcels
    if parc == "Schaefer200":
        conn_mdz = conn_mdz[:, :, :200, :200]
            
    # calculate mapconn
    mapconn_mdz = mapconn_from_connectivity(
        conn_mdz, 
        parc, 
        n_jobs=n_jobs, 
        run_null=False, 
        run_nointerhemispheric=False,
        measures=["pearson"],
        save_path=wd / "data_deriv" / "mapconn_pickled" / "drug_mdz" / f"parc-{parc}_window_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "drug_mdz" / f"parc-{parc}_window.csv.gz",
        index_levels=["treat"],
        overwrite=overwrite
    )

del conn_mdz, mapconn_mdz

Schaefer200Subcortical
connectivity_xarray.shape: (1, 4264, 216, 216)
xarray_dims: ('measure', 'sub-treat', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-treat', 'sub', 'treat', 'parcelx', 'parcely']
conn_measures: ['pearson']
retaining conn_measures: ['pearson']
index_to_iterate: Index(['placebo', 'midazolam'], dtype='object', name='treat')


Processing pearson:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)


pearson | all | idc = ('placebo',): (2296, 216, 216)


INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles
/Applications/miniforge3/envs/mapfc/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Calculating mapConn curves: 100%|██████████| 500/500 [00:15<00:00, 31.61it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentil

...

pearson | all | idc = ('placebo',): (2296, 200, 200)


INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:14<00:00, 35.39it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:14<00:00, 34.52it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<cl

Storing results
pearson | all | idc = ('midazolam',): (1968, 200, 200)


INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:10<00:00, 47.45it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:11<00:00, 45.22it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<cl

Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/drug_mdz/parc-Schaefer200_window_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/drug_mdz/parc-Schaefer200_window.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level treat (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 'placebo')
Getting statistics for: ('pearson', 'all', 'midazolam')


## YRSP

In [17]:
# iterate parcs
for parc in ["Schaefer200Subcortical", "Schaefer200"]:
    print(parc)
    
    # load connectivity
    conn_yrsp = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "yrsp" / f"parc-{parc}_connectivity.pkl.gz", "rb"))

    # calculate mapconn
    mapconn_yrsp = mapconn_from_connectivity(
        conn_yrsp, 
        parc, 
        n_jobs=n_jobs, 
        run_null=True,
        run_delta_null=False,
        n_perm=n_perm,
        run_nointerhemispheric=False,
        measures=["pearson"],
        save_path=wd / "data_deriv" / "mapconn_pickled" / "yrsp" / f"parc-{parc}_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "yrsp" / f"parc-{parc}.csv.gz",
        index_levels=["run"],
        drop_nulls=True,
        overwrite=overwrite
    )

del conn_yrsp, mapconn_yrsp

Schaefer200Subcortical
connectivity_xarray.shape: (1, 52, 216, 216)
xarray_dims: ('measure', 'sub-run', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-run', 'sub', 'run', 'parcelx', 'parcely']
conn_measures: ['pearson']
retaining conn_measures: ['pearson']
index_to_iterate: Index(['01', '02'], dtype='object', name='run')


Processing pearson:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles


pearson | all | idc = ('01',): (26, 216, 216)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 2223.77it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 2199.01it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

...

pearson | all | idc = ('01',): (26, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 4193.73it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 4113.74it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
pearson | all | idc = ('02',): (26, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3739.91it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 4464.01it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/yrsp/parc-Schaefer200_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/yrsp/parc-Schaefer200.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level run (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 1)
Getting statistics for: ('pearson', 'all', 2)


## HCP-EP

In [18]:
# PET
for parc in ["Schaefer200", "Schaefer200Subcortical"]:
    print(parc)
        
    # load connectivity
    conn_ep = pickle.load(gzip.open(wd / "data_deriv" / "connectomes" / "hcp_ep" / f"parc-{parc}_connectivity.pkl.gz", "rb"))
    
    # run
    mapconn_ep = mapconn_from_connectivity(  
        conn_ep, 
        parc, 
        n_jobs=n_jobs,
        run_null=True,
        run_delta_null=False,
        n_perm=n_perm,
        run_nointerhemispheric=False,
        measures=["pearson"],        
        drop_nulls=True,      
        save_path=wd / "data_deriv" / "mapconn_pickled" / "hcp_ep" / f"parc-{parc}_mapconn.pkl.gz",
        save_path_stats=wd / "results" / "neofc" / "hcp_ep" / f"parc-{parc}.csv.gz",
        index_levels=["dx"],
        overwrite=overwrite
    )
    
del mapconn_ep, conn_ep

Schaefer200
connectivity_xarray.shape: (1, 151, 200, 200)
xarray_dims: ('measure', 'sub-dx', 'parcelx', 'parcely')
xarray_coords: ['measure', 'sub-dx', 'sub', 'dx', 'parcelx', 'parcely']
conn_measures: ['pearson']
retaining conn_measures: ['pearson']
index_to_iterate: Index(['CTRL', 'SCZ'], dtype='object', name='dx')


Processing pearson:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:mapconn.mapconn:712: Flattening connectivity matrices to create MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles


pearson | all | idc = ('CTRL',): (55, 200, 200)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 3082.67it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 200 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 2641.37it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

...

pearson | all | idc = ('CTRL',): (55, 216, 216)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 2406.49it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 2607.08it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
pearson | all | idc = ('SCZ',): (96, 216, 216)


Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 1502.18it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:1337: Creating MapConnInv from MapConn instance (n_jobs=None, dtype=None)
INFO:mapconn.mapconn:618: Creating MapConn from flattened connectivity matrices (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.percentiles:44: Got map data with 25 maps and 216 values
INFO:mapconn.percentiles:63: Calculating map percentiles
Calculating mapConn curves: 100%|██████████| 500/500 [00:00<00:00, 1684.03it/s]
INFO:mapconn.mapconn:87: Initializing MapConn (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:898: Initializing MapConnInv (n_jobs=-1, dtype=<class 'numpy.float32'>)
INFO:mapconn.mapconn:2767: Creating MapConnNull instance from MapConnInv instance (n_nulls=1000, dype=None, n_jobs=-1)
INFO:mapconn.mapconn:2839: Using provided null maps
Calculating null mapconn curves: 100%|██████████| 100

Dropping nulls. Keeping null stats: True
Storing results
Saving results to: /Users/llotter/projects/mapfc/data_deriv/mapconn_pickled/hcp_ep/parc-Schaefer200Subcortical_mapconn.pkl.gz
Saving stats to: /Users/llotter/projects/mapfc/results/neofc/hcp_ep/parc-Schaefer200Subcortical.csv.gz (stat: auc, save_delta: False)
Iterating level measure (should be measure)
Iterating level connections (should be connections)
Iterating level dx (should be run, ses, or treat?)
Getting statistics for: ('pearson', 'all', 'CTRL')
Getting statistics for: ('pearson', 'all', 'SCZ')
